# 🚀 IBM LinuxONE Cloud Deployment & Acceleration via Google Colab

This Google Colab notebook acts as your remote bridge to deploy, configure, and benchmark the **FLare-AI** project on the **IBM LinuxONE (s390x) Virtual Machine** (`148.100.76.172`).

### What this notebook does:
1. **Authenticates** with IBM LinuxONE via SSH private key or password.
2. **Transfers** the FLare-AI codebase to `/home/linux1/FLare-Ai` (via Git clone/pull or SCP zip).
3. **Configures the s390x Environment** (installs `requirements-ibmz.txt`, sets up `snapml` with `--break-system-packages` and creates system symlinks).
4. **Verifies Telum Acceleration** (validates `nnpa` hardware flags).
5. **Runs the Pipeline & Benchmarks** (runs `snapml_benchmark.py` and `run_all.py --no-app`).

In [ ]:
# @title 1. Configuration & VM Credentials
LINUXONE_IP = "148.100.76.172"     # @param {type:"string"}
LINUXONE_USER = "linux1"           # @param {type:"string"}
REMOTE_DIR = "/home/linux1/FLare-Ai" # @param {type:"string"}
GITHUB_REPO = "https://github.com/thor11223344/Flare-AI.git" # @param {type:"string"}

print(f"Target Host: {LINUXONE_USER}@{LINUXONE_IP}")
print(f"Target Directory: {REMOTE_DIR}")

In [ ]:
# @title 2. SSH Authentication Setup (Key or Password)
# Choose whether you want to upload your SSH key file (linuxone_key) or use password/sshpass
import os

!apt-get install -y -qq sshpass openssh-client

USE_KEY = True # @param {type:"boolean"}
KEY_PATH = "/root/.ssh/linuxone_key"

os.makedirs("/root/.ssh", exist_ok=True)

if USE_KEY:
    if not os.path.exists(KEY_PATH):
        print("Please upload your 'linuxone_key' file:")
        import importlib; files = importlib.import_module("google.colab.files")
        uploaded = files.upload()
        key_name = list(uploaded.keys())[0]
        os.rename(key_name, KEY_PATH)
    !chmod 600 {KEY_PATH}
    SSH_CMD = f"ssh -o StrictHostKeyChecking=no -i {KEY_PATH} {LINUXONE_USER}@{LINUXONE_IP}"
    SCP_CMD = f"scp -o StrictHostKeyChecking=no -i {KEY_PATH}"
    print("SSH Key configured successfully!")
else:
    PASSWORD = input("Enter password for linux1: ")
    SSH_CMD = f"sshpass -p '{PASSWORD}' ssh -o StrictHostKeyChecking=no {LINUXONE_USER}@{LINUXONE_IP}"
    SCP_CMD = f"sshpass -p '{PASSWORD}' scp -o StrictHostKeyChecking=no"
    print("Password authentication configured.")

In [ ]:
# @title 3. Test Remote Connection
!{SSH_CMD} 'echo "[+] Connected successfully to IBM LinuxONE:"; uname -m; cat /etc/os-release | grep PRETTY_NAME'

In [ ]:
# @title 4. Deploy Codebase to LinuxONE
# Option A: Clone/Pull directly from GitHub to the VM (Recommended)
# Option B: SCP upload local files from Colab

DEPLOY_METHOD = "GitHub Clone/Pull" # @param ["GitHub Clone/Pull", "Upload ZIP from Colab"]

if DEPLOY_METHOD == "GitHub Clone/Pull":
    remote_script = f"""
    if [ -d '{REMOTE_DIR}/.git' ]; then
        echo 'Updating existing repository...'
        cd '{REMOTE_DIR}' && git pull
    else
        echo 'Cloning repository...'
        git clone {GITHUB_REPO} '{REMOTE_DIR}'
    fi
    """
    !{SSH_CMD} "{remote_script}"
else:
    print("Please upload project.zip (if not using Git):")
    import importlib; files = importlib.import_module("google.colab.files")
    up = files.upload()
    zip_name = list(up.keys())[0]
    !{SCP_CMD} {zip_name} {LINUXONE_USER}@{LINUXONE_IP}:/home/linux1/
    !{SSH_CMD} "unzip -o /home/linux1/{zip_name} -d {REMOTE_DIR}"

print("Deployment complete!")

In [ ]:
# @title 5. Install Dependencies & Snap ML on LinuxONE
install_cmd = f"""
cd '{REMOTE_DIR}'
echo '[1/3] Installing IBM Z dependencies from requirements-ibmz.txt...'
python3 -m pip install --break-system-packages -r requirements-ibmz.txt

echo '[2/3] Installing IBM Snap ML...'
python3 -m pip install --break-system-packages snapml

echo '[3/3] Creating kernel dist-packages symlink for Snap ML...'
sudo ln -sf /home/linux1/.local/lib/python3.12/site-packages/snapml /usr/local/lib/python3.12/dist-packages/snapml
echo '[✓] Environment setup completed.'
"""
!{SSH_CMD} "{install_cmd}"

In [ ]:
# @title 6. Verify IBM Telum Hardware Acceleration (nnpa)
verify_cmd = """
echo '--- Hardware Architecture ---'
lscpu | grep -E 'Architecture|Model name|CPU\(s\):'
echo '--- Telum Coprocessor Flag (nnpa) ---'
lscpu | grep -i nnpa
python3 -c "import snapml; print('[✓] Snap ML loaded successfully, version:', snapml.__version__)"
"""
!{SSH_CMD} "{verify_cmd}"

In [ ]:
# @title 7. Run Snap ML Hardware Benchmark on LinuxONE
!{SSH_CMD} "cd '{REMOTE_DIR}' && python3 src/ibmz/snapml_benchmark.py"

In [ ]:
# @title 8. Run Full FLare-AI Pipeline on LinuxONE
# Generates synthetic data, trains models, runs equity audit & governance log
!{SSH_CMD} "cd '{REMOTE_DIR}' && python3 run_all.py --no-app --force"

## 💡 Alternative: Connect Colab directly to the LinuxONE Jupyter Server

If you want to use the **Google Colab UI** while running code natively inside the IBM LinuxONE mainframe:
1. On LinuxONE, verify Jupyter is running:
   ```bash
   nohup jupyter notebook --ip=0.0.0.0 --port=8888 --no-browser --NotebookApp.token='datathon2026' > ~/jupyter.log 2>&1 &
   ```
2. On your local machine (Mac / Windows), start an SSH port forward:
   ```bash
   ssh -i ~/.ssh/linuxone_key -N -L 8888:localhost:8888 linux1@148.100.76.172
   ```
3. In Google Colab, click the **Connect** button (top right) ▾ -> **Connect to a local runtime**.
4. Enter the URL: `http://localhost:8888/?token=datathon2026` and click **Connect**.
5. Your Colab notebook cells will now execute directly on IBM z16 Telum processors!